In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import string
import math
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from efficientdet.data_pipeline.utils import xyxy_to_xywh, xywh_to_xyxy
from efficientdet.data_pipeline.anchors import AnchorBox
from efficientdet.data_pipeline.target_encoder import TargetEncoder
from efficientdet.data_pipeline.input_dataset import create_combined_dataset, create_images_dataset
from efficientdet.model.efficientdet import efficientdet
from efficientdet.model.losses import smooth_l1, focal

# From scratch

In [ ]:
model, model_prediction = efficientdet(phi=0)
model.summary()

# Train MNIST detection

## Load data

In [ ]:
path_train = '../data/train.csv'
path_test = '../data/test.csv'


def read_csv(path):
    df = pd.read_csv(path, dtype={'img_path': str, 'x1': 'int32', 'y1': 'int32', 'x2': 'int32', 'y2': 'int32', 'label': 'int32'})
    print(df.shape)
    return df


df_train = read_csv(path_train)
df_test = read_csv(path_test)

In [ ]:
ds_train = create_combined_dataset(path_train)
ds_test = create_combined_dataset(path_test)

In [ ]:
adam = tf.keras.optimizers.Adam()
losses = {'regression': smooth_l1(), 'classification': focal()}


def scheduler(epoch, lr):
    return lr * tf.math.exp(-0.1)

callbacks = [
    tf.keras.callbacks.LearningRateScheduler(scheduler)
]

model.compile(optimizer=adam, loss=losses)

In [ ]:
history = model.fit(ds_train, epochs=1, callbacks=callbacks)

In [ ]:
model.evaluate(ds_test)

# Test set predictions

In [ ]:
model.load_weights('../data/results/checkpoints/efficientdet-11.hdf5')

In [ ]:
from efficientdet.data_pipeline.decode_predictions import DecodePredictions

dp = DecodePredictions(num_classes=10, max_detections_per_class=10, confidence_threshold=0.5)

In [ ]:
ds_test = create_images_dataset(df_test)

In [ ]:
for img in ds_test.take(1):
    print(img.shape)

In [ ]:
img = tf.expand_dims(img, axis=0)

In [ ]:
for img, dct in ds_train.take(1):
    print(img.shape)

In [ ]:
pd.Series(np.ravel(dct['classification'].numpy()[2])).value_counts()

In [ ]:
model_out = model(img)

In [ ]:
preds = dp([img] + [model_out[1]] + [model_out[0]])

In [ ]:
[p.shape for p in preds]

In [ ]:
boxes, scores, classes = preds

In [ ]:
def add_box(ax, bbox, ec='r', fc='none', lw=0.5):
    x1, y1, x2, y2 = bbox
    height = y2 - y1
    width = x2 - x1
    rect = Rectangle((x1, y1), width, height, ec=ec, fc=fc, lw=lw)
    ax.add_patch(rect)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

ax.imshow(img[0])

for (box, score) in zip(boxes[0], scores[0]):
    if score > 0:
        add_box(ax, box)